## Enrichir les données APLs avec le temps d'accès aux urgences

In [16]:
import pandas as pd
from pathlib import Path

# On essaie d'utiliser __file__, sinon on prend le dossier actuel
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

ROOT = BASE_DIR.parent
DATA_DIR = ROOT / "data"

%ls ../data/extracted/
print('-'*30)
%ls ../data/processed/data/commune

communes_31_geom_simplified.parquet
communes_31_geom_simplified_geo.parquet
communes_densite.csv
dept_taux_mortalité_2024.parquet
mortalite_departements.csv
raw_comm_acces_urgences.csv
raw_comm_acces_urgences.parquet
raw_dept_taux_mortalité_2024.csv
raw_dept_taux_mortalité_2024.parquet
------------------------------
commune_clusters.parquet               raw_comm_score_apl_niv0.parquet
commune_niv1_score_irdes.csv           raw_comm_score_apl_niv1.parquet
commune_niv1_score_irdes.parquet       score_sante_territoires_final.csv
raw_comm_score_apl_niv0.csv            score_sante_territoires_final.parquet


In [17]:
# data/processed/data/commune/raw_comm_score_apl_niv1.csv
file_path = DATA_DIR / 'processed/data/commune/raw_comm_score_apl_niv1.parquet'

df_apl = pd.read_parquet(file_path)

df_apl.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,apl_infirmiers_std,apl_kines_std,apl_sagesfemmes_std,score_apl,quintile_apl_nat,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832.0,35.270,122.416,...,0.044420,-0.610034,1.186782,-0.317946,Q2 (faible),859.0,1590.0,01,200069193,84
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267.0,36.427,109.849,...,-0.140549,-0.510025,-0.214993,-0.291066,Q2 (faible),273.0,920.0,01,240100883,84
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854.0,59.621,201.121,...,1.202849,1.084761,1.284043,0.778222,Q5 (très bon),15554.0,2460.0,01,240100883,84
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897.0,50.539,131.876,...,0.183658,0.774416,1.587885,0.583430,Q5 (très bon),1917.0,1590.0,01,200042497,84
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113.0,10.929,44.227,...,-1.106414,-1.130735,-0.453044,-1.231006,Q1 (très faible),114.0,590.0,01,200040350,84


In [18]:
# data/extracted/raw_comm_acces_urgences.parquet

file_path = DATA_DIR / 'extracted/raw_comm_acces_urgences.parquet'

df_urg = pd.read_parquet(file_path)

df_urg.head()

,Code commune Insee,Libellé commune,Département,Région,Libellé région,SU,SMUR,MCS,HéliSMUR,HéliSC,Population 2014,typ_equip,tps_SU_SMUR,tps_SU_SMUR_MCS,tps_ychelismur,tps_ycheli,com_equipee,densite
0,01001,L' Abergement-Clémenciat,01,84,AUVERGNE-RHONE-ALPES,0,0,0,0,0,767,HeliSMUR,31.0,31.0,30.195652,30.195652,69029,49.009585
1,01002,L' Abergement-de-Varey,01,84,AUVERGNE-RHONE-ALPES,0,0,0,0,0,239,SU_SMUR,18.0,18.0,18.000000,18.000000,01004,26.206140
2,01004,Ambérieu-en-Bugey,01,84,AUVERGNE-RHONE-ALPES,1,1,0,0,0,14022,SU_SMUR,0.0,0.0,0.000000,0.000000,01004,572.794118
3,01005,Ambérieux-en-Dombes,01,84,AUVERGNE-RHONE-ALPES,0,0,0,0,0,1627,SU_SMUR,23.5,23.5,23.500000,23.500000,69013,101.370717
4,01006,Ambléon,01,84,AUVERGNE-RHONE-ALPES,0,0,0,0,0,109,SU_SMUR,15.0,15.0,15.000000,15.000000,01034,18.106312


In [19]:
df_urg.columns

Index(['Code commune Insee', 'Libellé commune', 'Département', 'Région',
       'Libellé région', 'SU', 'SMUR', 'MCS', 'HéliSMUR', 'HéliSC',
       'Population 2014', 'typ_equip', 'tps_SU_SMUR', 'tps_SU_SMUR_MCS',
       'tps_ychelismur', 'tps_ycheli', 'com_equipee', 'densite'],
      dtype='object')

In [21]:
df_merge = pd.merge(
    df_apl, 
    df_urg[['Code commune Insee', 'tps_SU_SMUR']], 
    left_on="code_insee_comm", 
    right_on="Code commune Insee",
    how="left"  # "left" pour garder toutes les lignes de df_apl
)

In [22]:
df_merge = df_merge.drop(columns=['Code commune Insee'])

In [23]:
df_merge.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,apl_kines_std,apl_sagesfemmes_std,score_apl,quintile_apl_nat,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region,tps_SU_SMUR
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832.0,35.270,122.416,...,-0.610034,1.186782,-0.317946,Q2 (faible),859.0,1590.0,01,200069193,84,31.0
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267.0,36.427,109.849,...,-0.510025,-0.214993,-0.291066,Q2 (faible),273.0,920.0,01,240100883,84,18.0
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854.0,59.621,201.121,...,1.084761,1.284043,0.778222,Q5 (très bon),15554.0,2460.0,01,240100883,84,0.0
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897.0,50.539,131.876,...,0.774416,1.587885,0.583430,Q5 (très bon),1917.0,1590.0,01,200042497,84,23.5
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113.0,10.929,44.227,...,-1.130735,-0.453044,-1.231006,Q1 (très faible),114.0,590.0,01,200040350,84,15.0


In [24]:
df_merge.shape

(34971, 35)

In [25]:
file_path = '../data/processed/data/commune/raw_comm_score_apl_niv2.parquet'
df_merge.to_parquet(file_path)